# Investigacion por IP con DHCP y DNS

## Objetivo

Reconstruir que dispositivo uso una IP y que actividad DNS genero durante una ventana de investigacion.

## Entradas esperadas

- `TargetIp`: IP investigada.
- `Lookback`: ventana de tiempo.

## Requisitos

- Acceso al area de trabajo de Microsoft Sentinel.
- Funciones KQL publicadas: `fn_Normalize_Windows_DHCP`, `fn_Normalize_Windows_DNS`, `fn_Correlate_DHCP_DNS`.
- Paquetes Python sugeridos: `msticpy`, `pandas`, `matplotlib`, `plotly`, `networkx` segun el notebook.

## Secciones

1. Historial DHCP.
2. Linea de tiempo DHCP + DNS.
3. Dominios consultados.
4. NXDOMAIN y senales anomalas.


In [ ]:
# Configuracion general - ajustar antes de ejecutar
workspace_id = "REEMPLAZAR_CON_WORKSPACE_ID"
tenant_id = "REEMPLAZAR_CON_TENANT_ID"

# Conexion sugerida con MSTICPy
# import msticpy as mp
# mp.init_notebook(namespace=globals())
# qry_prov = mp.QueryProvider("MSSentinel")
# qry_prov.connect(workspace=workspace_id, tenant_id=tenant_id)


In [ ]:
query_dhcp = """
let TargetIp = "REEMPLAZAR_CON_IP";
let Lookback = 7d;
fn_Normalize_Windows_DHCP(Lookback)
| where ClientIp == TargetIp
| project TimeGenerated, DeviceName, EventAction, ClientIp, ClientMac, HostName, ScopeId, RawMessage
| order by TimeGenerated asc
"""
# dhcp_df = qry_prov.exec_query(query_dhcp)
print(query_dhcp)


In [ ]:
query_timeline = """
let TargetIp = "REEMPLAZAR_CON_IP";
let Lookback = 7d;
let dhcp_timeline =
    fn_Normalize_Windows_DHCP(Lookback)
    | where ClientIp == TargetIp
    | project TimeGenerated, Source = "DHCP", DeviceName, ClientIp, HostName, ClientMac, Detail = strcat(EventAction, " scope=", ScopeId), RawMessage;
let dns_timeline =
    fn_Correlate_DHCP_DNS(Lookback)
    | where ClientIp == TargetIp
    | project TimeGenerated, Source = "DNS", DeviceName = DnsServer, ClientIp, HostName, ClientMac, Detail = strcat(QueryType, " ", QueryName, " respuesta=", ResponseCode), RawMessage;
union dhcp_timeline, dns_timeline
| order by TimeGenerated asc
"""
# timeline_df = qry_prov.exec_query(query_timeline)
print(query_timeline)


In [ ]:
query_dns_summary = """
let TargetIp = "REEMPLAZAR_CON_IP";
let Lookback = 7d;
fn_Correlate_DHCP_DNS(Lookback)
| where ClientIp == TargetIp
| summarize TotalQueries = count(), DistinctDomains = dcount(QueryRootDomain), NxdomainCount = countif(toupper(ResponseCode) has_any ("NXDOMAIN", "NAME_ERROR", "3") or RawMessage has_any ("NXDOMAIN", "Name Error")), TxtQueries = countif(toupper(QueryType) == "TXT"), MaxQueryLength = max(QueryLength), MaxLabelLength = max(MaxLabelLength), SampleQueries = make_set(QueryName, 20) by ClientIp, HostName, ClientMac
"""
# dns_summary_df = qry_prov.exec_query(query_dns_summary)
print(query_dns_summary)


## Resumen para incidente

Documentar aqui:

- Hallazgos principales.
- Entidades relevantes: IP, hostname, direccion MAC, dominio.
- Evidencia KQL usada.
- Recomendacion: cerrar, monitorear, escalar o contener.
